In [1]:
import pandas as pd

In [2]:
ARABIZI_TO_ARABIC = {
    # Digits
    "2": "ء",
    "3": "ع",
    "5": "خ",
    "6": "ط",
    "7": "ح",
    "8": "ه",
    "9": "ق",

    # Digraphs
    "ch": "ش",
    "sh": "ش",
    "kh": "خ",
    "gh": "غ",
    "dh": "ذ",
    "th": "ث",
    "ou": "و",
    "oo": "و",
    "aa": "ا",
    "ee": "ي",
    "ii": "ي",

    # Latin characters
    "a": "ا",
    "b": "ب",
    "c": "ك",
    "d": "د",
    "e": "",
    "f": "ف",
    "g": "گ",
    "h": "ه",
    "i": "ي",
    "j": "ج",
    "k": "ك",
    "l": "ل",
    "m": "م",
    "n": "ن",
    "o": "و",
    "p": "پ",
    "q": "ق",
    "r": "ر",
    "s": "س",
    "t": "ت",
    "u": "و",
    "v": "ڤ",
    "w": "و",
    "x": "كس",
    "y": "ي",
    "z": "ز"
}

In [3]:
def naive_mapping_transliterate(text, mapping):
    text = text.lower()
    result = []
    i = 0
    keys = sorted(mapping, key=len, reverse=True)

    while i < len(text):
        match = next(
            (key for key in keys if text.startswith(key, i)),
            None
        )

        if match is not None:
            result.append(mapping[match])
            i += len(match)
        else:
            result.append(text[i])
            i += 1

    return "".join(result)

In [4]:
from pathlib import Path
import sys

# Support running the notebook from either the project root or notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Use validation for development; reserve the test split for final evaluation.
df = pd.read_csv(project_root / "data/processed/val.csv")
df["prediction"] = df["Darija_latin"].apply(
    lambda text: naive_mapping_transliterate(text, ARABIZI_TO_ARABIC)
)
df[["Darija_latin", "Darija_arabic", "prediction"]].head()


,Darija_latin,Darija_arabic,prediction
0,bayna homa tay7awlo ib9aw mbrrdin.,باينا هوما تايحاولو إبقاو مبرّدين.,باينا هوما تايحاولو يبقاو مبرردين.
1,"ssimana jjaya, nhar lkhmis","السيمانا الجايا, نهار لخميس","سسيمانا ججايا, نهار لخميس"
2,ewa bda l9raya bkri!,إوا بدا لقرايا بكري!,وا بدا لقرايا بكري!
3,tan8Dr 3la mrrakch,تانهضر علا مرّاكش,تانهدر علا مرراكش
4,bjojna tankhafo mn TTayaran,بجوجنا تانخافو من الطاياران,بجوجنا تانخافو من تتاياران


In [5]:
from scripts.metrics import evaluate_dataframe

scores = evaluate_dataframe(
    df,
    reference_col="Darija_arabic",
    prediction_col="prediction",
)
print(scores)


{'example_count': 985, 'cer': 0.08805184285135682, 'wer': 0.3115376032135684, 'exact_match': 0.2467005076142132, 'character_errors': 2174, 'reference_characters': 24690, 'word_errors': 1396, 'reference_words': 4481, 'exact_matches': 243}
